# Data Science & Database Assignment: Session 10
## Topic: JOINS - Part 2 (FULL OUTER JOIN, SELF JOIN, Multi-Table Chained JOINs & Query Optimization)

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `joins_part2.db`  
**Core Concepts:** `FULL OUTER JOIN`, `SELF JOIN` (Hierarchical Trees), Multi-Table Chained `LEFT JOIN`s, Handling Duplicates in 1:N Relationships (`DISTINCT` vs `GROUP BY`), Join Condition Optimization (Integer PK vs String Natural Keys)  
**Status:** Completed & Pre-Executed  

---

### Key Concepts Overview
In this session, we advance beyond basic binary joins to cover complex relational patterns:
1. **FULL OUTER JOIN**: Combines `LEFT JOIN` and `RIGHT JOIN` into a single bidirectional result set, preserving unmatched rows from both sides with `NULL` padding.
2. **SELF JOIN**: Joining a table with itself to represent hierarchical parent-child relationships (e.g. Spotify playlist folders, employee managers, product category trees).
3. **Chained Multi-Table JOINs**: Connecting three or more tables sequentially (`Table A -> Table B -> Table C`) while preserving primary records at every stage using `LEFT JOIN`.
4. **Duplicate Row Resolution**: Diagnosing and eliminating duplicate parent rows caused by 1:N cardinality using `DISTINCT` or `GROUP BY` aggregations.
5. **Join Condition Efficiency**: Analyzing CPU complexity, B-Tree index branching factors, and hash join mechanics between numeric surrogate keys and alphanumeric strings.


## Task 1: FULL JOIN on Influencers & Collaborations
> **Problem Statement:** Create two tables: `Influencers` (`id`, `name`) and `Collaborations` (`id`, `influencer1_id`, `influencer2_id`, `collab_date`). Write a SQL `FULL JOIN` query to list all influencers and show their collaboration partner names if any, including influencers with no collaborations.

### Conceptual Architecture
- A `FULL OUTER JOIN` retains:
  1. Influencers that participated in collaborations (e.g., *Virat Kohli*, *Anushka Sharma*, *CarryMinati*, *Bhuvan Bam*).
  2. Solo influencers who have **zero collaborations** (e.g., *Technical Guruji*, *Prajakta Koli*), showing `NULL` for collaboration fields.
  3. External or unlinked collaboration entries (e.g., collaboration ID 104 with unknown IDs 99 & 100), showing `NULL` for influencer attributes.


In [1]:
import sqlite3
import pandas as pd

# Connect to SQLite database
conn = sqlite3.connect('joins_part2.db')
cursor = conn.cursor()

# Reset tables for clean reproducibility
cursor.execute("DROP TABLE IF EXISTS Collaborations;")
cursor.execute("DROP TABLE IF EXISTS Influencers;")

cursor.execute("""
CREATE TABLE Influencers (
    id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Collaborations (
    id INT PRIMARY KEY,
    influencer1_id INT,
    influencer2_id INT,
    collab_date DATE NOT NULL
);
""")

# Populate Influencers
influencers_data = [
    (1, 'Virat Kohli'),
    (2, 'Anushka Sharma'),
    (3, 'CarryMinati'),
    (4, 'Bhuvan Bam'),
    (5, 'Technical Guruji'),  # Solo creator (No collabs)
    (6, 'Prajakta Koli')       # Solo creator (No collabs)
]
cursor.executemany("INSERT INTO Influencers VALUES (?, ?);", influencers_data)

# Populate Collaborations
collaborations_data = [
    (101, 1, 2, '2026-01-15'),  # Virat & Anushka
    (102, 3, 4, '2026-02-10'),  # CarryMinati & Bhuvan Bam
    (103, 1, 3, '2026-03-01'),  # Virat & CarryMinati
    (104, 99, 100, '2026-04-05') # Unlinked external collab
]
cursor.executemany("INSERT INTO Collaborations VALUES (?, ?, ?, ?);", collaborations_data)
conn.commit()

# Execute FULL JOIN Query:
task1_query = """
SELECT 
    i1.id AS influencer_id,
    i1.name AS influencer_name,
    c.id AS collab_id,
    c.collab_date,
    i2.name AS partner_name
FROM Influencers i1
FULL JOIN Collaborations c ON i1.id = c.influencer1_id
LEFT JOIN Influencers i2 ON c.influencer2_id = i2.id;
"""

df_t1 = pd.read_sql_query(task1_query, conn)
display(df_t1)
print(f"Total rows in FULL JOIN: {len(df_t1)}")


,influencer_id,influencer_name,collab_id,collab_date,partner_name
0,1.0,Virat Kohli,101.0,2026-01-15,Anushka Sharma
1,1.0,Virat Kohli,103.0,2026-03-01,CarryMinati
2,2.0,Anushka Sharma,NaN,NaN,NaN
3,3.0,CarryMinati,102.0,2026-02-10,Bhuvan Bam
4,4.0,Bhuvan Bam,NaN,NaN,NaN
5,5.0,Technical Guruji,NaN,NaN,NaN
6,6.0,Prajakta Koli,NaN,NaN,NaN
7,NaN,NaN,104.0,2026-04-05,NaN


Total rows in FULL JOIN: 8


## Task 2: SELF JOIN on Playlists (Spotify Nested Playlists)
> **Problem Statement:** Using a `SELF JOIN`, write a query on a table called `Playlists` (`id`, `user_id`, `playlist_name`, `parent_playlist_id`) to display each playlist alongside its parent playlist name, similar to how Spotify shows nested playlists.  
> *Hint: Join Playlists with itself on parent_playlist_id = id.*

### Conceptual Architecture
In hierarchical relational schemas, an entity points to itself:
- A playlist folder (e.g., `Workout Motivation`) has `parent_playlist_id = NULL` (Root).
- A child playlist (e.g., `High Intensity Cardio`) has `parent_playlist_id = 2` pointing back to the parent folder.
- By performing a `LEFT JOIN Playlists parent ON child.parent_playlist_id = parent.id`, every playlist is displayed, with root items cleanly showing `'None (Root Playlist)'` via `COALESCE()`.


In [2]:
cursor.execute("DROP TABLE IF EXISTS Playlists;")

cursor.execute("""
CREATE TABLE Playlists (
    id INT PRIMARY KEY,
    user_id INT NOT NULL,
    playlist_name VARCHAR(100) NOT NULL,
    parent_playlist_id INT
);
""")

playlists_data = [
    (1, 101, 'Top Hits 2026', None),             # Root
    (2, 101, 'Workout Motivation', None),         # Root Folder
    (3, 101, 'High Intensity Cardio', 2),         # Nested Child of 2
    (4, 101, 'Heavy Weightlifting', 2),           # Nested Child of 2
    (5, 102, 'Chill & Acoustic', None),           # Root Folder
    (6, 102, 'Sunday Morning Coffee', 5),         # Nested Child of 5
    (7, 102, 'Rainy Day Acoustic', 5)             # Nested Child of 5
]
cursor.executemany("INSERT INTO Playlists VALUES (?, ?, ?, ?);", playlists_data)
conn.commit()

# Query: All playlists with parent folder name
task2_query = """
SELECT 
    child.id AS playlist_id,
    child.playlist_name,
    child.parent_playlist_id,
    COALESCE(parent.playlist_name, 'None (Root Playlist)') AS parent_playlist_name
FROM Playlists child
LEFT JOIN Playlists parent ON child.parent_playlist_id = parent.id;
"""

df_t2 = pd.read_sql_query(task2_query, conn)
display(df_t2)


,playlist_id,playlist_name,parent_playlist_id,parent_playlist_name
0,1,Top Hits 2026,NaN,None (Root Playlist)
1,2,Workout Motivation,NaN,None (Root Playlist)
2,3,High Intensity Cardio,2.0,Workout Motivation
3,4,Heavy Weightlifting,2.0,Workout Motivation
4,5,Chill & Acoustic,NaN,None (Root Playlist)
5,6,Sunday Morning Coffee,5.0,Chill & Acoustic
6,7,Rainy Day Acoustic,5.0,Chill & Acoustic


## Task 3: Multiple JOINs across 3 Tables (Users, Orders, Payments)
> **Problem Statement:** Given three tables: `Users` (`id`, `username`), `Orders` (`id`, `user_id`, `order_date`), and `Payments` (`id`, `order_id`, `amount`), write a SQL query using multiple JOINs to display each username, their order date, and payment amount, showing all users even if they have no orders or payments.

### Conceptual Architecture
To preserve all users regardless of their downstream activity, we chain **`LEFT JOIN`** operations:
1. `Users u LEFT JOIN Orders o ON u.id = o.user_id` preserves users without orders (`priya_m`).
2. `LEFT JOIN Payments p ON o.id = p.order_id` preserves orders that do not yet have recorded payments (`rahul_v`).


In [3]:
cursor.execute("DROP TABLE IF EXISTS Payments;")
cursor.execute("DROP TABLE IF EXISTS Orders;")
cursor.execute("DROP TABLE IF EXISTS Users;")

cursor.execute("""
CREATE TABLE Users (
    id INT PRIMARY KEY,
    username VARCHAR(50) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Orders (
    id INT PRIMARY KEY,
    user_id INT NOT NULL,
    order_date DATE NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Payments (
    id INT PRIMARY KEY,
    order_id INT NOT NULL,
    amount DECIMAL(10, 2) NOT NULL
);
""")

users_data = [(1, 'atharv_c'), (2, 'sneha_p'), (3, 'rahul_v'), (4, 'priya_m')]
orders_data = [(101, 1, '2026-03-01'), (102, 2, '2026-03-05'), (103, 2, '2026-03-10'), (104, 3, '2026-03-12')]
payments_data = [(1001, 101, 750.00), (1002, 102, 1200.00), (1003, 103, 350.00)]

cursor.executemany("INSERT INTO Users VALUES (?, ?);", users_data)
cursor.executemany("INSERT INTO Orders VALUES (?, ?, ?);", orders_data)
cursor.executemany("INSERT INTO Payments VALUES (?, ?, ?);", payments_data)
conn.commit()

# Multi-Table Chained LEFT JOIN Query:
task3_query = """
SELECT 
    u.id AS user_id,
    u.username,
    o.id AS order_id,
    o.order_date,
    p.id AS payment_id,
    p.amount
FROM Users u
LEFT JOIN Orders o ON u.id = o.user_id
LEFT JOIN Payments p ON o.id = p.order_id;
"""

df_t3 = pd.read_sql_query(task3_query, conn)
display(df_t3)


,user_id,username,order_id,order_date,payment_id,amount
0,1,atharv_c,101.0,2026-03-01,1001.0,750.0
1,2,sneha_p,102.0,2026-03-05,1002.0,1200.0
2,2,sneha_p,103.0,2026-03-10,1003.0,350.0
3,3,rahul_v,104.0,2026-03-12,NaN,NaN
4,4,priya_m,NaN,NaN,NaN,NaN


## Task 4: Eliminating Duplicates in One-to-Many Relationships (Zomato Restaurants & Reviews)
> **Problem Statement:** You notice that your JOIN query between Zomato's Restaurants and Reviews tables is returning duplicate rows for some restaurants. Modify your query to eliminate duplicates and explain in one line why the duplicates were happening.  
> *Hint: Use DISTINCT or GROUP BY and consider the relationship between restaurants and reviews.*

### One-Line Explanation of Why Duplicates Occur:
> **"Duplicates occur because the relationship between Restaurants and Reviews is One-to-Many (1:N), causing the parent restaurant row to duplicate for every matching review row."**

### Elimination Techniques:
1. **`DISTINCT`**: Filters the final result set to unique restaurant records.
2. **`GROUP BY`**: Summarizes child records into valuable aggregate metrics (e.g. `COUNT(review_id)`, `AVG(rating)`).


In [4]:
cursor.execute("DROP TABLE IF EXISTS Reviews;")
cursor.execute("DROP TABLE IF EXISTS Restaurants;")

cursor.execute("""
CREATE TABLE Restaurants (
    id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    city VARCHAR(50) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Reviews (
    id INT PRIMARY KEY,
    restaurant_id INT NOT NULL,
    user_name VARCHAR(50) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL,
    review_text VARCHAR(255)
);
""")

restaurants_data = [
    (1, 'Truffles', 'Bangalore'),
    (2, 'Peter Cat', 'Kolkata'),
    (3, 'Haldiram', 'Nagpur'),
    (4, 'Barbeque Nation', 'Mumbai')
]
reviews_data = [
    (1, 1, 'Amit', 4.5, 'Amazing burgers!'),
    (2, 1, 'Rohit', 4.8, 'Best peri peri fries.'),
    (3, 1, 'Pooja', 4.2, 'Great ambience.'),
    (4, 2, 'Sagnik', 4.9, 'Chelo Kebab is legendary.'),
    (5, 2, 'Debolina', 4.7, 'Classic Kolkata nostalgia.'),
    (6, 3, 'Tanmay', 4.0, 'Nice chole bhature.')
]
cursor.executemany("INSERT INTO Restaurants VALUES (?, ?, ?);", restaurants_data)
cursor.executemany("INSERT INTO Reviews VALUES (?, ?, ?, ?, ?);", reviews_data)
conn.commit()

print("--- 1. Query with Duplicates (Notice Truffles repeated 3 times) ---")
display(pd.read_sql_query("""
SELECT r.id AS restaurant_id, r.name AS restaurant_name, r.city 
FROM Restaurants r 
INNER JOIN Reviews rev ON r.id = rev.restaurant_id;
""", conn))

print("--- 2. Solution using DISTINCT ---")
display(pd.read_sql_query("""
SELECT DISTINCT r.id AS restaurant_id, r.name AS restaurant_name, r.city 
FROM Restaurants r 
INNER JOIN Reviews rev ON r.id = rev.restaurant_id;
""", conn))

print("--- 3. Solution using GROUP BY (Analytics Summary) ---")
display(pd.read_sql_query("""
SELECT 
    r.id AS restaurant_id,
    r.name AS restaurant_name,
    r.city,
    COUNT(rev.id) AS total_reviews,
    ROUND(AVG(rev.rating), 2) AS avg_rating
FROM Restaurants r
LEFT JOIN Reviews rev ON r.id = rev.restaurant_id
GROUP BY r.id, r.name, r.city;
""", conn))


--- 1. Query with Duplicates (Notice Truffles repeated 3 times) ---


,restaurant_id,restaurant_name,city
0,1,Truffles,Bangalore
1,1,Truffles,Bangalore
2,1,Truffles,Bangalore
3,2,Peter Cat,Kolkata
4,2,Peter Cat,Kolkata
5,3,Haldiram,Nagpur


--- 2. Solution using DISTINCT ---


,restaurant_id,restaurant_name,city
0,1,Truffles,Bangalore
1,2,Peter Cat,Kolkata
2,3,Haldiram,Nagpur


--- 3. Solution using GROUP BY (Analytics Summary) ---


,restaurant_id,restaurant_name,city,total_reviews,avg_rating
0,1,Truffles,Bangalore,3,4.5
1,2,Peter Cat,Kolkata,2,4.8
2,3,Haldiram,Nagpur,1,4.0
3,4,Barbeque Nation,Mumbai,0,NaN


## Task 5: Comparing Two JOIN Conditions on Products & Categories (Flipkart Model)
> **Problem Statement:** Write two different JOIN queries on a `Products` and `Categories` table (like Flipkart) to list all products with their category names, but use different join conditions in each. Briefly explain which join condition is more efficient and why.

---

### Efficiency Analysis

#### Query 1: Join on Integer Surrogate Key (`p.category_id = c.category_id`)
```sql
SELECT p.product_id, p.product_name, p.price, c.category_name
FROM Products p
INNER JOIN Categories c ON p.category_id = c.category_id;
```

#### Query 2: Join on String Natural Key (`p.category_code = c.category_code`)
```sql
SELECT p.product_id, p.product_name, p.price, c.category_name
FROM Products p
INNER JOIN Categories c ON p.category_code = c.category_code;
```

#### Why Condition 1 (Integer Key) is Far More Efficient:
1. **CPU & ALU Architecture:** Integer equality comparisons are 4-byte or 8-byte fixed-width scalar operations executed in a single CPU instruction clock cycle. String comparisons require byte-by-byte lexicographical scanning with locale collation checks.
2. **Index Density & Memory Efficiency:** Numeric B-Trees have a much higher branching factor than string indexes. More keys fit into each 8KB memory page, minimizing cache misses and disk I/O operations.
3. **Hash Join Partitioning:** Query optimizers perform fast bitwise hashing on integers with negligible hash collision probability compared to variable-length character strings.


In [5]:
cursor.execute("DROP TABLE IF EXISTS Products;")
cursor.execute("DROP TABLE IF EXISTS Categories;")

cursor.execute("""
CREATE TABLE Categories (
    category_id INT PRIMARY KEY,
    category_code VARCHAR(20) UNIQUE NOT NULL,
    category_name VARCHAR(100) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    category_id INT,
    category_code VARCHAR(20)
);
""")

categories_data = [
    (1, 'ELEC', 'Electronics'),
    (2, 'FASH', 'Fashion & Apparel'),
    (3, 'HOME', 'Home & Kitchen')
]
products_data = [
    (101, 'MacBook Air M3', 114900.00, 1, 'ELEC'),
    (102, 'Sony WH-1000XM5', 28990.00, 1, 'ELEC'),
    (103, "Levi's 511 Slim Jeans", 2999.00, 2, 'FASH'),
    (104, 'Cotton Casual Shirt', 1499.00, 2, 'FASH'),
    (105, 'Philips Air Fryer', 8499.00, 3, 'HOME')
]
cursor.executemany("INSERT INTO Categories VALUES (?, ?, ?);", categories_data)
cursor.executemany("INSERT INTO Products VALUES (?, ?, ?, ?, ?);", products_data)
conn.commit()

print("--- Join Query 1: Integer Key (p.category_id = c.category_id) ---")
display(pd.read_sql_query("""
SELECT 
    p.product_id,
    p.product_name,
    p.price,
    c.category_name
FROM Products p
INNER JOIN Categories c ON p.category_id = c.category_id;
""", conn))

print("--- Join Query 2: String Key (p.category_code = c.category_code) ---")
display(pd.read_sql_query("""
SELECT 
    p.product_id,
    p.product_name,
    p.price,
    c.category_name
FROM Products p
INNER JOIN Categories c ON p.category_code = c.category_code;
""", conn))

conn.close()


--- Join Query 1: Integer Key (p.category_id = c.category_id) ---


,product_id,product_name,price,category_name
0,101,MacBook Air M3,114900,Electronics
1,102,Sony WH-1000XM5,28990,Electronics
2,103,Levi's 511 Slim Jeans,2999,Fashion & Apparel
3,104,Cotton Casual Shirt,1499,Fashion & Apparel
4,105,Philips Air Fryer,8499,Home & Kitchen


--- Join Query 2: String Key (p.category_code = c.category_code) ---


,product_id,product_name,price,category_name
0,101,MacBook Air M3,114900,Electronics
1,102,Sony WH-1000XM5,28990,Electronics
2,103,Levi's 511 Slim Jeans,2999,Fashion & Apparel
3,104,Cotton Casual Shirt,1499,Fashion & Apparel
4,105,Philips Air Fryer,8499,Home & Kitchen


## Summary & Practical Engineering Guidelines

1. **`FULL OUTER JOIN`**: Essential for reconciliation and synchronization between two systems where either side can introduce independent updates.
2. **`SELF JOIN`**: The relational gold standard for modeling single-table hierarchies (e.g., categories & subcategories, employee reporting lines, playlist folder nesting).
3. **Multi-Table `LEFT JOIN` Chaining**: Ensure the root parent table is placed first, chaining child tables with consistent foreign key conditions to prevent premature record truncation.
4. **De-duplication in 1:N Relationships**:
   - Use `DISTINCT` when you simply need unique entity identities.
   - Use `GROUP BY` when you can enrich the query with aggregate business metrics (e.g. review counts, average ratings).
5. **Join Keys**: Always establish joins on indexed integer surrogate keys (`INT / BIGINT`) rather than descriptive character strings for optimal query throughput.
